# Phase 4 — Model Training & *Unseen-Drug* Validation
## Grouped cross-validation of four regressors against EE%

**Input:** `data/processed/PLGA_clean_unscaled.csv` (unscaled, from Phase 2).
**Outputs:** `results/tables/ML_grouped_performance_metrics.csv` (leaderboard) and the best fitted pipeline in `src/models/`.

### Why the *unscaled* file is the correct input
Phase 2 also produced `ML_ready_PLGA.csv`, in which `StandardScaler` was fit on all 430 rows. Feeding that into cross-validation would leak test-fold means/SDs into training. Here we start from the **unscaled** table and put `StandardScaler` **inside** the pipeline, so it is re-fit on the training rows of each fold only. This is the leakage-proof path flagged in the Phase 2 caveat.

### What this notebook does
1. Merge the severely imbalanced LA/GA grades `1.86`, `2.33`, `5.67` into a single `other` level; re-apply one-hot encoding.
2. Define `GroupKFold(n_splits=5)` on `drug_group` — **no drug appears in both train and test**.
3. Build four `Pipeline` objects, each with `StandardScaler` as the first step.
4. Run the grouped CV loop, recording R², MAE and RMSE per fold.
5. Aggregate mean ± SD into a leaderboard; persist the lowest-mean-MAE pipeline with `joblib`.

**No SHAP / explainability is run here.** No hyperparameter search is performed — these are honest untuned baselines (tuning belongs to a later phase and must be nested *inside* the grouped CV).

In [1]:
# --- Setup ---
import json, platform
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import sklearn
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.dummy import DummyRegressor
from sklearn.compose import TransformedTargetRegressor
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error
import xgboost as xgb
from xgboost import XGBRegressor

pd.set_option("display.width", 220); pd.set_option("display.max_columns", 60)

def find_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "data" / "raw").is_dir():
            return p
    raise RuntimeError("Could not locate project root (folder containing data/raw).")

ROOT   = find_root(Path.cwd())
PROC   = ROOT / "data" / "processed"
TAB    = ROOT / "results" / "tables"
MODELS = ROOT / "src" / "models"
for d in (TAB, MODELS): d.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
N_SPLITS     = 5

print("Project root :", ROOT)
print("Models ->", MODELS)
print(f"python {platform.python_version()} | scikit-learn {sklearn.__version__} | xgboost {xgb.__version__} | pandas {pd.__version__}")

Project root : C:\Users\Ali sheroz\Desktop\PLGA-EE-Generalization
Models -> C:\Users\Ali sheroz\Desktop\PLGA-EE-Generalization\src\models
python 3.12.10 | scikit-learn 1.9.0 | xgboost 3.4.1 | pandas 3.0.5


## 1. Load the unscaled dataset and merge the imbalanced LA/GA grades
Phase 3 showed the copolymer grades are severely unbalanced: `1` (n=349), `3` (n=76), but `1.86` (n=3), `2.33` (n=1), `5.67` (n=1). Three one-hot columns backed by 1–3 formulations are effectively noise, and under `GroupKFold` they are absent from most training folds entirely — a near-constant column that exists in test but not train.

We therefore merge `{1.86, 2.33, 5.67}` into a single `other` level, giving `LAGA_grp ∈ {1, 3, other}`.

**Honest limitation:** `other` still totals only **5 formulations**, so any coefficient or split on it rests on almost no evidence. Merging stops it from *breaking* the CV; it does not make those grades well-sampled. Grade-specific conclusions must not be drawn for `other`.

In [2]:
df = pd.read_csv(PROC / "PLGA_clean_unscaled.csv")
print("Loaded PLGA_clean_unscaled.csv:", df.shape)

# Leakage guard: these must not exist in the modelling input at all.
for bad in ("LC", "particle_size"):
    assert bad not in df.columns, f"LEAKAGE: {bad} present in modelling input — abort."
print("Leakage guard PASS: LC and particle_size absent from the modelling input.")

RARE_GRADES = [1.86, 2.33, 5.67]
before = df["LA/GA"].value_counts().sort_index()

df["LAGA_grp"] = np.where(df["LA/GA"].isin(RARE_GRADES), "other",
                          df["LA/GA"].map(lambda x: f"{x:g}"))
after = df["LAGA_grp"].value_counts()

regroup = (pd.DataFrame({"original_LA_GA": before.index, "n_formulations": before.to_numpy()})
             .assign(merged_into=lambda t: np.where(t["original_LA_GA"].isin(RARE_GRADES), "other",
                                                    t["original_LA_GA"].map(lambda x: f"{x:g}"))))
regroup["n_drug_groups"] = [int(df.loc[df["LA/GA"] == g, "drug_group"].nunique()) for g in regroup["original_LA_GA"]]
regroup.to_csv(TAB / "laga_regrouping.csv", index=False)
display(regroup)
print("LA/GA levels after merge:")
print(after.to_string())
print(f"\n'other' spans {int(df.loc[df['LAGA_grp']=='other','drug_group'].nunique())} distinct drug groups "
      f"out of {df['drug_group'].nunique()} — sparse by construction.")

Loaded PLGA_clean_unscaled.csv: (430, 26)
Leakage guard PASS: LC and particle_size absent from the modelling input.


,original_LA_GA,n_formulations,merged_into,n_drug_groups
0,1.00,349,1,57
1,1.86,3,other,3
2,2.33,1,other,1
3,3.00,76,3,8
4,5.67,1,other,1


LA/GA levels after merge:
LAGA_grp
1        349
3         76
other      5

'other' spans 4 distinct drug groups out of 63 — sparse by construction.


## 2. Assemble the design matrix
- **Continuous (13):** scaled *inside* each fold by the pipeline's `StandardScaler`.
- **Categorical → one-hot (7):** `pH_cat` (`-1`, `0`, `1`, `missing`) and `LAGA_grp` (`1`, `3`, `other`).

One-hot encoding is applied here, outside the CV loop, using an **explicit fixed category list**. This is deliberate and is *not* leakage: one-hot encoding is a deterministic schema mapping that estimates no statistic from the data (unlike scaling, which learns a mean and SD and therefore must stay inside the fold). Fixing the categories also guarantees every fold sees the same 20 columns in the same order, even when a rare level is absent from a training fold.

`EE` is the sole target. Metadata (`row_id`, drug/study identity, `EE_is_zero`) is never a feature; `drug_group` is used only as the CV grouping key.

In [3]:
CONTINUOUS = ["mol_MW","mol_logP","mol_TPSA","mol_melting_point","mol_Hacceptors","mol_Hdonors",
              "mol_heteroatoms","polymer_MW","drug/polymer","surfactant_concentration",
              "aqueous/organic","surfactant_HLB","solvent_polarity_index"]
PH_LEVELS   = ["-1", "0", "1", "missing"]
LAGA_LEVELS = ["1", "3", "other"]
TARGET      = "EE"

ph   = pd.Categorical(df["pH_cat"].astype(str),   categories=PH_LEVELS)
laga = pd.Categorical(df["LAGA_grp"].astype(str), categories=LAGA_LEVELS)
assert not pd.isna(ph).any() and not pd.isna(laga).any(), "Unexpected category outside the fixed level list."

dummies = pd.concat([
    pd.get_dummies(ph,   prefix="pH").astype(float),
    pd.get_dummies(laga, prefix="LAGA").astype(float),
], axis=1)
dummies.index = df.index

X       = pd.concat([df[CONTINUOUS], dummies], axis=1)
y       = df[TARGET].to_numpy(float)
groups  = df["drug_group"].to_numpy()
FEATURES = list(X.columns)

assert X.isna().sum().sum() == 0 and np.isfinite(y).all()
print(f"X: {X.shape[0]} formulations x {X.shape[1]} features "
      f"({len(CONTINUOUS)} continuous + {dummies.shape[1]} one-hot)")
print("features:", FEATURES)
print(f"\ny = {TARGET}: mean {y.mean():.2f}, SD {y.std(ddof=1):.2f}, range {y.min():.1f}-{y.max():.1f}")
print(f"groups: {len(np.unique(groups))} drug groups")

X: 430 formulations x 20 features (13 continuous + 7 one-hot)


features: ['mol_MW', 'mol_logP', 'mol_TPSA', 'mol_melting_point', 'mol_Hacceptors', 'mol_Hdonors', 'mol_heteroatoms', 'polymer_MW', 'drug/polymer', 'surfactant_concentration', 'aqueous/organic', 'surfactant_HLB', 'solvent_polarity_index', 'pH_-1', 'pH_0', 'pH_1', 'pH_missing', 'LAGA_1', 'LAGA_3', 'LAGA_other']

y = EE: mean 64.79, SD 23.51, range 0.0-98.9
groups: 63 drug groups


## 3. Leakage-proof validation: `GroupKFold(n_splits=5)` on `drug_group`
Every formulation of a given drug (including its salt/hydrate variants, which Phase 2 collapsed into one `drug_group`) lands entirely in either the training or the test side of a fold. Each test fold therefore measures performance on drugs the model has **never seen**, which is the research question.

We assert this explicitly — a silent grouping bug would invalidate the whole study, so it is checked rather than assumed. We also verify no raw `small_molecule_name` straddles a split, which catches any failure of the salt/hydrate collapse.

In [4]:
gkf = GroupKFold(n_splits=N_SPLITS)
splits = list(gkf.split(X, y, groups=groups))

rows = []
for k, (tr, te) in enumerate(splits, 1):
    g_tr, g_te = set(groups[tr]), set(groups[te])
    n_tr, n_te = set(df["small_molecule_name"].to_numpy()[tr]), set(df["small_molecule_name"].to_numpy()[te])
    assert not (g_tr & g_te), f"Fold {k}: drug_group leaked across the split!"
    assert not (n_tr & n_te), f"Fold {k}: small_molecule_name leaked across the split!"
    rows.append({"fold": k, "n_train": len(tr), "n_test": len(te),
                 "train_groups": len(g_tr), "test_groups": len(g_te),
                 "test_EE_mean": y[te].mean(), "test_EE_SD": y[te].std(ddof=1),
                 "test_has_LAGA_other": bool(X.iloc[te]["LAGA_other"].sum() > 0),
                 "train_has_LAGA_other": bool(X.iloc[tr]["LAGA_other"].sum() > 0)})
fold_info = pd.DataFrame(rows)
fold_info.to_csv(TAB / "ML_cv_fold_composition.csv", index=False)
display(fold_info.round(2))
print("Leakage assertions PASS for all folds: no drug_group and no small_molecule_name "
      "appears on both sides of any split.")
print("\nNote the varying test_EE_mean across folds: because entire drugs move together, folds are")
print("not exchangeable samples. Per-fold R2 will therefore be volatile and can go negative.")

,fold,n_train,n_test,train_groups,test_groups,test_EE_mean,test_EE_SD,test_has_LAGA_other,train_has_LAGA_other
0,1,344,86,51,12,76.84,17.82,True,True
1,2,344,86,50,13,58.14,29.84,True,True
2,3,344,86,50,13,60.31,18.47,False,True
3,4,344,86,51,12,67.74,18.07,True,True
4,5,344,86,50,13,60.90,26.03,False,True


Leakage assertions PASS for all folds: no drug_group and no small_molecule_name appears on both sides of any split.

Note the varying test_EE_mean across folds: because entire drugs move together, folds are
not exchangeable samples. Per-fold R2 will therefore be volatile and can go negative.


## 4. Model pipelines
`StandardScaler` is the first step of every pipeline, so the mean/SD are learned from that fold's training rows only. Scaling the one-hot columns as well is harmless: the tree ensembles are invariant to monotone rescaling, and `LinearRegression`/`SVR` benefit from a common scale.

**Settings are library defaults** (only `random_state`/`n_jobs` are pinned for reproducibility). No hyperparameter search was run, so these are untuned baselines.

**One correctness exception — SVR.** `SVR`'s `epsilon` (0.1) and `C` (1.0) defaults are expressed in *target* units. With EE spanning 0–98.9 (SD ≈ 23.5), an untouched SVR cannot reach the response scale and would collapse toward a constant — a measurement artefact, not a property of SVR. We therefore wrap it in `TransformedTargetRegressor(transformer=StandardScaler())`, which standardises `y` **using training-fold statistics only** and inverts the prediction afterwards. This is target scaling as a specification requirement (the counterpart of feature scaling), not selective tuning.

A `DummyRegressor` (predicts the training-fold mean) is included as a clearly-labelled **reference**, not as one of the four requested models: it makes explicit whether a model beats simply guessing the average EE.

In [5]:
def make_models():
    """Fresh, unfitted pipelines. StandardScaler is always step 1."""
    return {
        "Linear Regression (baseline)": Pipeline([
            ("scaler", StandardScaler()),
            ("model", LinearRegression()),
        ]),
        "Random Forest": Pipeline([
            ("scaler", StandardScaler()),
            ("model", RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1)),
        ]),
        "XGBoost": Pipeline([
            ("scaler", StandardScaler()),
            ("model", XGBRegressor(random_state=RANDOM_STATE, n_jobs=-1,
                                   tree_method="hist", verbosity=0)),
        ]),
        "SVR (RBF)": Pipeline([
            ("scaler", StandardScaler()),
            ("model", TransformedTargetRegressor(regressor=SVR(),
                                                 transformer=StandardScaler())),
        ]),
        "DummyRegressor (reference)": Pipeline([
            ("scaler", StandardScaler()),
            ("model", DummyRegressor(strategy="mean")),
        ]),
    }

MODEL_ORDER = list(make_models().keys())
for name, pipe in make_models().items():
    assert pipe.steps[0][0] == "scaler" and isinstance(pipe.steps[0][1], StandardScaler)
    print(f"{name:<30} -> {pipe.steps[-1][1].__class__.__name__}")
print("\nAll pipelines verified: StandardScaler is step 1 in each (re-fit per fold).")

Linear Regression (baseline)   -> LinearRegression
Random Forest                  -> RandomForestRegressor
XGBoost                        -> XGBRegressor
SVR (RBF)                      -> TransformedTargetRegressor
DummyRegressor (reference)     -> DummyRegressor

All pipelines verified: StandardScaler is step 1 in each (re-fit per fold).


## 5. Grouped training loop
For each model and each fold: fit on the training drugs, predict the held-out **unseen** drugs, and record R², MAE and RMSE. We also stitch together the out-of-fold (OOF) predictions, because every formulation is held out exactly once — pooled OOF metrics are far more stable than averaging volatile per-fold R² values, and both are reported.

In [6]:
per_fold, oof_pred = [], {name: np.full(len(y), np.nan) for name in MODEL_ORDER}

for name in MODEL_ORDER:
    for k, (tr, te) in enumerate(splits, 1):
        pipe = make_models()[name]                      # fresh, unfitted
        pipe.fit(X.iloc[tr], y[tr])                     # scaler fit on TRAIN rows only
        pred = pipe.predict(X.iloc[te])
        oof_pred[name][te] = pred
        per_fold.append({
            "model": name, "fold": k,
            "n_train": len(tr), "n_test": len(te), "test_groups": len(set(groups[te])),
            "R2":   r2_score(y[te], pred),
            "MAE":  mean_absolute_error(y[te], pred),
            "RMSE": root_mean_squared_error(y[te], pred),
        })
    print(f"  done: {name}")

fold_metrics = pd.DataFrame(per_fold)
for name, p in oof_pred.items():
    assert np.isfinite(p).all(), f"{name}: OOF predictions incomplete."
fold_metrics.to_csv(TAB / "ML_grouped_per_fold_metrics.csv", index=False)

print("\nPer-fold metrics (every test fold contains only unseen drugs):")
display(fold_metrics.pivot(index="fold", columns="model", values="MAE").round(2)
        .rename_axis(columns="MAE by fold"))
display(fold_metrics.pivot(index="fold", columns="model", values="R2").round(3)
        .rename_axis(columns="R2 by fold"))

  done: Linear Regression (baseline)


  done: Random Forest


  done: XGBoost


  done: SVR (RBF)
  done: DummyRegressor (reference)

Per-fold metrics (every test fold contains only unseen drugs):


MAE by fold,DummyRegressor (reference),Linear Regression (baseline),Random Forest,SVR (RBF),XGBoost
fold,,,,,
1,21.05,17.50,17.65,16.04,24.87
2,23.97,20.86,20.19,20.25,23.45
3,14.63,30.21,14.03,19.32,20.35
4,14.60,23.29,17.78,19.65,14.97
5,22.03,31.74,13.82,16.40,16.78


R2 by fold,DummyRegressor (reference),Linear Regression (baseline),Random Forest,SVR (RBF),XGBoost
fold,,,,,
1,-0.723,-0.437,-0.654,-0.174,-1.761
2,-0.079,0.209,0.318,0.291,0.188
3,-0.093,-2.290,-0.043,-0.765,-1.047
4,-0.042,-1.300,-0.504,-0.628,-0.139
5,-0.035,-1.638,0.503,0.294,0.297


## 6. Aggregate leaderboard
Mean ± SD across the 5 folds, plus pooled out-of-fold metrics. Ranked by **mean MAE (ascending)**, which is the criterion for selecting the pipeline to persist.

In [7]:
agg = (fold_metrics.groupby("model")[["R2", "MAE", "RMSE"]]
       .agg(["mean", "std"]))
agg.columns = [f"{m}_{s}" for m, s in agg.columns]
agg = agg.reset_index()

# pooled out-of-fold metrics (each formulation predicted exactly once, always as an unseen drug)
pooled = pd.DataFrame([{
    "model": name,
    "OOF_R2":   r2_score(y, oof_pred[name]),
    "OOF_MAE":  mean_absolute_error(y, oof_pred[name]),
    "OOF_RMSE": root_mean_squared_error(y, oof_pred[name]),
} for name in MODEL_ORDER])

leaderboard = (agg.merge(pooled, on="model")
               .sort_values("MAE_mean")
               .reset_index(drop=True))
leaderboard.insert(0, "rank_by_mean_MAE", np.arange(1, len(leaderboard) + 1))
leaderboard["is_reference_not_a_candidate"] = leaderboard["model"].str.contains("Dummy")
leaderboard["n_splits"] = N_SPLITS
leaderboard["cv_strategy"] = f"GroupKFold(n_splits={N_SPLITS}) on drug_group"
leaderboard["target"] = TARGET
leaderboard["n_formulations"] = len(y)
leaderboard["n_drug_groups"] = int(len(np.unique(groups)))

ORDER = ["rank_by_mean_MAE","model","R2_mean","R2_std","MAE_mean","MAE_std","RMSE_mean","RMSE_std",
         "OOF_R2","OOF_MAE","OOF_RMSE","is_reference_not_a_candidate",
         "cv_strategy","n_splits","target","n_formulations","n_drug_groups"]
leaderboard = leaderboard[ORDER]

LB_PATH = TAB / "ML_grouped_performance_metrics.csv"
leaderboard.to_csv(LB_PATH, index=False)
display(leaderboard.drop(columns=["cv_strategy","n_splits","target","n_formulations","n_drug_groups"]).round(4))
print(">>> Leaderboard SAVED to:", LB_PATH)

,rank_by_mean_MAE,model,R2_mean,R2_std,MAE_mean,MAE_std,RMSE_mean,RMSE_std,OOF_R2,OOF_MAE,OOF_RMSE,is_reference_not_a_candidate
0,1,Random Forest,-0.0758,0.5021,16.6954,2.7215,21.2615,2.6873,0.1697,16.6954,21.3970,False
1,2,SVR (RBF),-0.1964,0.4970,18.3311,1.9624,22.6477,2.3089,0.0620,18.3311,22.7417,False
2,3,DummyRegressor (reference),-0.1943,0.2965,19.2568,4.3689,23.5864,5.1626,-0.0476,19.2568,24.0342,True
3,4,XGBoost,-0.4924,0.8844,20.0868,4.2251,24.6624,4.1416,-0.1280,20.0868,24.9391,False
4,5,Linear Regression (baseline),-1.0912,0.9869,24.7202,6.0925,30.0422,7.9544,-0.7286,24.7202,30.8732,False


>>> Leaderboard SAVED to: C:\Users\Ali sheroz\Desktop\PLGA-EE-Generalization\results\tables\ML_grouped_performance_metrics.csv


## 7. Robustness check — repeated grouped CV over 5 shuffles
`GroupKFold` without shuffling is deterministic, so a single run gives one arbitrary partition of the 63 drug groups. Phase 1 recommended *repeated* grouped CV. We repeat with `shuffle=True` across 5 seeds (25 fits per model) to check whether the ranking is stable or an artefact of one partition.

In [8]:
rep_rows = []
for seed in range(5):
    gkf_s = GroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    sp = list(gkf_s.split(X, y, groups=groups))
    for tr, te in sp:
        assert not (set(groups[tr]) & set(groups[te])), "group leak in repeated CV"
    for name in MODEL_ORDER:
        for k, (tr, te) in enumerate(sp, 1):
            pipe = make_models()[name]
            pipe.fit(X.iloc[tr], y[tr])
            pred = pipe.predict(X.iloc[te])
            rep_rows.append({"model": name, "seed": seed, "fold": k,
                             "R2": r2_score(y[te], pred),
                             "MAE": mean_absolute_error(y[te], pred),
                             "RMSE": root_mean_squared_error(y[te], pred)})
rep = pd.DataFrame(rep_rows)
rep_agg = (rep.groupby("model")[["R2","MAE","RMSE"]].agg(["mean","std"]))
rep_agg.columns = [f"{m}_{s}" for m, s in rep_agg.columns]
rep_agg = rep_agg.reset_index().sort_values("MAE_mean").reset_index(drop=True)
rep_agg.insert(0, "rank_by_mean_MAE", np.arange(1, len(rep_agg) + 1))
rep_agg["n_fits_per_model"] = rep["seed"].nunique() * N_SPLITS
rep_agg.to_csv(TAB / "ML_grouped_performance_repeated.csv", index=False)
display(rep_agg.round(4))

single = leaderboard.set_index("model")["rank_by_mean_MAE"]
repeat = rep_agg.set_index("model")["rank_by_mean_MAE"]
cmp = pd.DataFrame({"rank_single_GroupKFold": single, "rank_repeated_5seeds": repeat})
cmp["rank_changed"] = cmp["rank_single_GroupKFold"] != cmp["rank_repeated_5seeds"]
display(cmp)
print("Ranking stable across the two schemes." if not cmp["rank_changed"].any()
      else "WARNING: ranking is partition-dependent — treat the single-split ordering with caution.")

,rank_by_mean_MAE,model,R2_mean,R2_std,MAE_mean,MAE_std,RMSE_mean,RMSE_std,n_fits_per_model
0,1,DummyRegressor (reference),-0.1080,0.1845,19.8238,3.5686,24.3424,4.3531,25
1,2,Random Forest,-0.4546,1.8913,20.4753,7.8139,24.9588,8.2412,25
2,3,SVR (RBF),-0.3862,0.9181,21.0691,4.3506,25.5306,4.2194,25
3,4,XGBoost,-0.6011,2.0876,21.3242,8.2565,26.0792,8.4868,25
4,5,Linear Regression (baseline),-0.9586,0.9761,25.9004,4.3072,31.0127,5.9445,25


,rank_single_GroupKFold,rank_repeated_5seeds,rank_changed
model,,,
DummyRegressor (reference),3,1,True
Linear Regression (baseline),5,5,False
Random Forest,1,2,True
SVR (RBF),2,3,True
XGBoost,4,4,False


## 8. Is the apparent advantage real? Size-independent pooled out-of-fold check
Section 7 flagged that the ranking changes between the single deterministic partition and 5 shuffled ones. Before reporting any winner we must establish whether that is a genuine difference or an artefact, because two things differ between the schemes:

- Un-shuffled `GroupKFold` **balances fold sizes** (here: exactly 86 test rows per fold). With `shuffle=True` the sizes swing from 46 to 143.
- Averaging *per-fold* MAE over unequal folds is an **unweighted** mean, which over-weights small folds.

The size-independent comparison is the **pooled out-of-fold MAE**: for each seed, every one of the 430 formulations is predicted exactly once while its drug is held out, then metrics are computed on the full assembled vector. Fold sizes then cannot bias the estimate. We repeat over 5 seeds and report mean ± SD.

This is the decisive test of whether the descriptors carry *transferable* signal to unseen drugs.

In [9]:
pool_rows = []
for seed in range(5):
    sp = list(GroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed).split(X, y, groups=groups))
    for name in MODEL_ORDER:
        oof = np.full(len(y), np.nan)
        for tr, te in sp:
            pipe = make_models()[name]
            pipe.fit(X.iloc[tr], y[tr])
            oof[te] = pipe.predict(X.iloc[te])
        assert np.isfinite(oof).all()
        pool_rows.append({"model": name, "seed": seed,
                          "OOF_MAE":  mean_absolute_error(y, oof),
                          "OOF_RMSE": root_mean_squared_error(y, oof),
                          "OOF_R2":   r2_score(y, oof)})
pool = pd.DataFrame(pool_rows)
pool_agg = pool.groupby("model")[["OOF_MAE","OOF_RMSE","OOF_R2"]].agg(["mean","std"])
pool_agg.columns = [f"{a}_{b}" for a, b in pool_agg.columns]
pool_agg = pool_agg.reset_index().sort_values("OOF_MAE_mean").reset_index(drop=True)
pool_agg.insert(0, "rank_by_pooled_OOF_MAE", np.arange(1, len(pool_agg) + 1))
pool_agg.to_csv(TAB / "ML_grouped_performance_pooled_oof_repeated.csv", index=False)
pool.to_csv(TAB / "ML_grouped_pooled_oof_per_seed.csv", index=False)

print("Pooled out-of-fold metrics, mean +/- SD over 5 shuffled grouped partitions:")
display(pool_agg.round(3))
print("Pooled OOF MAE per seed (lower is better):")
display(pool.pivot(index="seed", columns="model", values="OOF_MAE").round(2))

# --- The decisive comparison, stated explicitly ---
dum = pool_agg.loc[pool_agg["model"].str.contains("Dummy")].iloc[0]
cand = pool_agg.loc[~pool_agg["model"].str.contains("Dummy")]
best_pool = cand.iloc[0]
wins = {}
piv = pool.pivot(index="seed", columns="model", values="OOF_MAE")
dum_col = [c for c in piv.columns if "Dummy" in c][0]
for c in piv.columns:
    if c != dum_col:
        wins[c] = int((piv[c] < piv[dum_col]).sum())

print("\n" + "=" * 74)
print("DECISIVE CHECK: does any model beat the mean-predictor on UNSEEN drugs?")
print("-" * 74)
print(f"mean-predictor          : pooled OOF MAE {dum['OOF_MAE_mean']:.2f} +/- {dum['OOF_MAE_std']:.2f}"
      f" | R2 {dum['OOF_R2_mean']:+.3f}")
print(f"best learned model      : {best_pool['model']} -> {best_pool['OOF_MAE_mean']:.2f}"
      f" +/- {best_pool['OOF_MAE_std']:.2f} | R2 {best_pool['OOF_R2_mean']:+.3f}")
print(f"\nseeds (out of 5) in which each model beats the mean-predictor on pooled OOF MAE:")
for k, v in sorted(wins.items(), key=lambda kv: -kv[1]):
    print(f"   {k:<32} {v}/5")
beats = best_pool["OOF_MAE_mean"] < dum["OOF_MAE_mean"]
print("-" * 74)
if beats:
    print("VERDICT: the best model beats the mean-predictor on average across partitions.")
else:
    print("VERDICT: NO model beats the mean-predictor on average across partitions.")
    print("On unseen drugs, the 20 descriptors do not carry reliably transferable signal at")
    print("this sample size. The mean-predictor is also far more STABLE")
    print(f"(SD {dum['OOF_MAE_std']:.2f} vs {best_pool['OOF_MAE_std']:.2f} EE%), i.e. the learned")
    print("models sometimes help and sometimes hurt depending on which drugs are held out.")
    print("This is a negative result about generalisation, not a bug, and must be reported as such.")
print("=" * 74)

Pooled out-of-fold metrics, mean +/- SD over 5 shuffled grouped partitions:


,rank_by_pooled_OOF_MAE,model,OOF_MAE_mean,OOF_MAE_std,OOF_RMSE_mean,OOF_RMSE_std,OOF_R2_mean,OOF_R2_std
0,1,DummyRegressor (reference),19.138,0.278,23.858,0.245,-0.032,0.021
1,2,Random Forest,20.710,3.181,26.387,5.002,-0.299,0.515
2,3,SVR (RBF),20.923,1.725,25.752,2.209,-0.210,0.215
3,4,XGBoost,21.862,4.224,27.690,5.702,-0.438,0.619
4,5,Linear Regression (baseline),25.717,1.421,31.322,1.965,-0.785,0.224


Pooled OOF MAE per seed (lower is better):


model,DummyRegressor (reference),Linear Regression (baseline),Random Forest,SVR (RBF),XGBoost
seed,,,,,
0,18.83,24.92,21.37,19.84,22.89
1,19.04,27.90,18.24,21.34,20.50
2,19.59,26.00,25.94,23.68,28.69
3,19.09,25.66,19.61,20.51,17.96
4,19.14,24.11,18.39,19.25,19.28



DECISIVE CHECK: does any model beat the mean-predictor on UNSEEN drugs?
--------------------------------------------------------------------------
mean-predictor          : pooled OOF MAE 19.14 +/- 0.28 | R2 -0.032
best learned model      : Random Forest -> 20.71 +/- 3.18 | R2 -0.299

seeds (out of 5) in which each model beats the mean-predictor on pooled OOF MAE:
   Random Forest                    2/5
   XGBoost                          1/5
   Linear Regression (baseline)     0/5
   SVR (RBF)                        0/5
--------------------------------------------------------------------------
VERDICT: NO model beats the mean-predictor on average across partitions.
On unseen drugs, the 20 descriptors do not carry reliably transferable signal at
this sample size. The mean-predictor is also far more STABLE
(SD 0.28 vs 3.18 EE%), i.e. the learned
models sometimes help and sometimes hurt depending on which drugs are held out.
This is a negative result about generalisation, not a bug, and

## 9. Persist the best pipeline (lowest mean MAE)
The winner is chosen by **lowest mean MAE** across the 5 `GroupKFold` folds, excluding the `DummyRegressor` reference. It is then re-fit on **all 430 formulations** (the standard deployment refit) and written to `src/models/` with `joblib`, alongside a metadata sidecar recording the feature order, CV scheme, and scores so the artefact is self-describing.

> **Read this before using the artefact.** Selection uses the single `GroupKFold` partition, as specified. Section 8 showed that this partition's ranking does **not** replicate across shuffled partitions. The saved model is therefore "the best of four on the specified split", **not** a validated predictor of EE% for new drugs. Its metadata records both the single-split scores and the pooled-OOF caveat.

In [10]:
candidates = leaderboard.loc[~leaderboard["is_reference_not_a_candidate"]]
best_name  = candidates.iloc[0]["model"]
best_row   = candidates.iloc[0]
dummy_row  = leaderboard.loc[leaderboard["is_reference_not_a_candidate"]].iloc[0]

print(f"Best of the four candidate models by mean MAE: {best_name}")
print(f"   mean MAE  {best_row['MAE_mean']:.3f} +/- {best_row['MAE_std']:.3f} EE%")
print(f"   mean R2   {best_row['R2_mean']:.3f} +/- {best_row['R2_std']:.3f}")
print(f"   mean RMSE {best_row['RMSE_mean']:.3f} +/- {best_row['RMSE_std']:.3f}")
print(f"   pooled OOF: R2 {best_row['OOF_R2']:.3f} | MAE {best_row['OOF_MAE']:.3f} | RMSE {best_row['OOF_RMSE']:.3f}")
print(f"\nMean-predictor reference ({dummy_row['model']}): "
      f"MAE {dummy_row['MAE_mean']:.3f}, OOF R2 {dummy_row['OOF_R2']:.3f}")
beat = best_row["MAE_mean"] < dummy_row["MAE_mean"]
print(f"Best model beats the mean-predictor on MAE: {beat} "
      f"(improvement {dummy_row['MAE_mean'] - best_row['MAE_mean']:+.3f} EE%, "
      f"{100*(dummy_row['MAE_mean']-best_row['MAE_mean'])/dummy_row['MAE_mean']:+.1f}%)")

best_pipe = make_models()[best_name]
best_pipe.fit(X, y)                      # final refit on the full dataset
slug = (best_name.lower().replace(" ", "_").replace("(", "").replace(")", "")
        .replace("/", "-"))
MODEL_PATH = MODELS / f"best_model_{slug}.joblib"
joblib.dump(best_pipe, MODEL_PATH)

sidecar = {
    "artifact": MODEL_PATH.name,
    "selected_model": best_name,
    "selection_criterion": "lowest mean MAE across GroupKFold(n_splits=5) on drug_group",
    "fitted_on": "all 430 formulations (final refit after model selection)",
    "target": TARGET, "target_units": "EE %  (unscaled)",
    "feature_order": FEATURES,
    "continuous_features": CONTINUOUS,
    "onehot_features": [c for c in FEATURES if c not in CONTINUOUS],
    "laga_grades_merged_into_other": RARE_GRADES,
    "leakage_excluded": ["LC", "particle_size"],
    "cv": {"strategy": f"GroupKFold(n_splits={N_SPLITS})", "group_key": "drug_group",
           "n_drug_groups": int(len(np.unique(groups)))},
    "cv_scores": {"R2_mean": float(best_row["R2_mean"]), "R2_std": float(best_row["R2_std"]),
                  "MAE_mean": float(best_row["MAE_mean"]), "MAE_std": float(best_row["MAE_std"]),
                  "RMSE_mean": float(best_row["RMSE_mean"]), "RMSE_std": float(best_row["RMSE_std"]),
                  "OOF_R2": float(best_row["OOF_R2"]), "OOF_MAE": float(best_row["OOF_MAE"]),
                  "OOF_RMSE": float(best_row["OOF_RMSE"])},
    "hyperparameters": "library defaults (no search performed); random_state=42",
    "generalisation_caveat": (
        "Selected on the single deterministic GroupKFold partition. Across 5 SHUFFLED grouped "
        "partitions the pooled out-of-fold MAE of every model was WORSE on average than a "
        "training-mean DummyRegressor. Do NOT present this artefact as a validated predictor of "
        "EE% for unseen drugs. See results/tables/ML_grouped_performance_pooled_oof_repeated.csv."
    ),
    "pooled_oof_repeated": {
        "mean_predictor_OOF_MAE": float(dum["OOF_MAE_mean"]),
        "best_model_OOF_MAE": float(best_pool["OOF_MAE_mean"]),
        "best_model_name": str(best_pool["model"]),
        "any_model_beats_mean_predictor_on_average": bool(beats),
    },
    "versions": {"python": platform.python_version(), "scikit-learn": sklearn.__version__,
                 "xgboost": xgb.__version__, "pandas": pd.__version__, "numpy": np.__version__},
}
(MODELS / f"best_model_{slug}.meta.json").write_text(json.dumps(sidecar, indent=2), encoding="utf-8")

# Prove the artefact round-trips and reproduces predictions
reloaded = joblib.load(MODEL_PATH)
assert np.allclose(reloaded.predict(X), best_pipe.predict(X)), "Reloaded model does not reproduce predictions."
print(f"\n>>> Best pipeline SAVED to: {MODEL_PATH}")
print(f">>> Metadata sidecar     : {(MODELS / f'best_model_{slug}.meta.json').name}")
print("Round-trip verified: the reloaded artefact reproduces predictions exactly.")

Best of the four candidate models by mean MAE: Random Forest
   mean MAE  16.695 +/- 2.721 EE%
   mean R2   -0.076 +/- 0.502
   mean RMSE 21.262 +/- 2.687
   pooled OOF: R2 0.170 | MAE 16.695 | RMSE 21.397

Mean-predictor reference (DummyRegressor (reference)): MAE 19.257, OOF R2 -0.048
Best model beats the mean-predictor on MAE: True (improvement +2.561 EE%, +13.3%)



>>> Best pipeline SAVED to: C:\Users\Ali sheroz\Desktop\PLGA-EE-Generalization\src\models\best_model_random_forest.joblib
>>> Metadata sidecar     : best_model_random_forest.meta.json
Round-trip verified: the reloaded artefact reproduces predictions exactly.


In [11]:
# --- Final summary ---
print("=" * 78)
print("PHASE 4 LEADERBOARD - unseen-drug GroupKFold CV (5 folds, grouped by drug_group)")
print("-" * 78)
disp = leaderboard[["rank_by_mean_MAE","model","MAE_mean","MAE_std","RMSE_mean","RMSE_std",
                    "R2_mean","R2_std","OOF_R2"]].copy()
disp.columns = ["#","model","MAE","MAE_SD","RMSE","RMSE_SD","R2","R2_SD","OOF_R2"]
print(disp.to_string(index=False, float_format=lambda v: f"{v:8.3f}"))
print("-" * 78)
print(f"target            : {TARGET} (%), sole target; LC and particle_size excluded")
print(f"data              : {len(y)} formulations, {len(np.unique(groups))} drug groups, "
      f"{df['reference'].nunique()} studies")
print(f"features          : {len(FEATURES)} ({len(CONTINUOUS)} continuous scaled in-fold + "
      f"{len(FEATURES)-len(CONTINUOUS)} one-hot)")
print(f"saved leaderboard : results/tables/ML_grouped_performance_metrics.csv")
print(f"saved best model  : src/models/{MODEL_PATH.name}  ({best_name})")
print("-" * 78)
print("ROBUSTNESS (5 shuffled grouped partitions, pooled out-of-fold, size-independent):")
print(f"   mean-predictor  MAE {dum['OOF_MAE_mean']:6.2f} +/- {dum['OOF_MAE_std']:.2f} | R2 {dum['OOF_R2_mean']:+.3f}")
print(f"   best model      MAE {best_pool['OOF_MAE_mean']:6.2f} +/- {best_pool['OOF_MAE_std']:.2f} | R2 {best_pool['OOF_R2_mean']:+.3f}  ({best_pool['model']})")
print(f"   any model beats the mean-predictor on average: {beats}")
print("   => the single-partition ranking does NOT replicate; treat the saved model as a")
print("      best-of-four artefact, not a validated unseen-drug predictor.")
print("SHAP / explainability: NOT run (deferred by instruction).")
print("=" * 78)

PHASE 4 LEADERBOARD - unseen-drug GroupKFold CV (5 folds, grouped by drug_group)
------------------------------------------------------------------------------
 #                        model      MAE   MAE_SD     RMSE  RMSE_SD       R2    R2_SD   OOF_R2
 1                Random Forest   16.695    2.721   21.262    2.687   -0.076    0.502    0.170
 2                    SVR (RBF)   18.331    1.962   22.648    2.309   -0.196    0.497    0.062
 3   DummyRegressor (reference)   19.257    4.369   23.586    5.163   -0.194    0.296   -0.048
 4                      XGBoost   20.087    4.225   24.662    4.142   -0.492    0.884   -0.128
 5 Linear Regression (baseline)   24.720    6.092   30.042    7.954   -1.091    0.987   -0.729
------------------------------------------------------------------------------
target            : EE (%), sole target; LC and particle_size excluded
data              : 430 formulations, 63 drug groups, 59 studies
features          : 20 (13 continuous scaled in-fold + 

## Phase 4 complete

Trained and validated four pipelines under leakage-proof, drug-grouped cross-validation. The leaderboard is at `results/tables/ML_grouped_performance_metrics.csv`; the lowest-mean-MAE pipeline is persisted in `src/models/`.

### The headline result is a negative one, and it is the honest one
On the single specified `GroupKFold` partition, Random Forest looks like a clear winner (mean MAE ≈ 16.7 EE% vs 19.3 for the mean-predictor). That advantage **does not survive** re-partitioning. Across 5 shuffled grouped partitions, using the size-independent pooled out-of-fold MAE, **no model beat a training-mean `DummyRegressor` on average** — and the mean-predictor was by far the most *stable* (SD ≈ 0.3 EE% versus ≈ 3.2 for Random Forest).

Interpretation: with 430 formulations over 63 drug groups, 51 % of drugs represented by a single formulation, and 20 descriptors, **the models are largely learning drug-specific and study-specific idiosyncrasies rather than transferable structure–encapsulation relationships.** When an entire drug is withheld, that memorisation cannot transfer, so performance collapses to — or below — the level of guessing the dataset average.

This is a legitimate and reportable scientific finding: it directly answers the project's research question ("how accurately can ML predict EE% for drugs excluded from training?") with "not reliably better than the dataset mean, on this dataset." It also explains why the literature's high random-split R² values are not evidence of unseen-drug generalisation — random splits let formulations of the same drug appear on both sides.

### Three constraints on any claim made from these numbers
1. **Per-fold R² is volatile** because whole drugs move between folds; folds are not exchangeable samples. Judge on pooled OOF metrics, not a single fold.
2. **No hyperparameter tuning was performed.** These are untuned baselines. Tuning may narrow the gap but must be nested *inside* the grouped CV, or it silently reintroduces selection leakage. Untuned results cannot be cited as an upper bound on achievable performance.
3. **The `other` LA/GA level holds 5 formulations** across 4 drug groups. Merging kept the CV from breaking; it did not make those grades well-sampled.

### Sensible next steps (not run here)
Nested hyperparameter tuning inside the grouped CV; a study-level (`GroupKFold` on `reference`) check to separate drug identity from study fingerprint — Phase 1 flag K, since 62/65 drugs appear in only one publication; restricting claims to the well-populated logP 0–5 region; and reporting per-drug-group errors to identify which chemistry fails.

**No SHAP or explainability analysis was run**, per instruction.